
# Uplift Modeling / Causal ML — Campanha de Clientes

Este notebook demonstra, com **dados sintéticos**, como estimar quais clientes são mais propensos a **mudar seu comportamento por causa de uma campanha**.

A ideia central é comparar:

- **Propensity modeling:** quem provavelmente compra?
- **Uplift modeling:** para quem a campanha aumenta a probabilidade de compra?

Será usado um **T-Learner**, uma abordagem de meta-learning causal que treina dois modelos separados: um para clientes tratados e outro para clientes de controle.

> **Importante:** os dados são simulados. O objetivo é aprender a metodologia, não produzir um modelo comercial real.


## 1. Instalação e imports

In [ ]:

# Se estiver no Google Colab e alguma biblioteca faltar:
# !pip install scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

np.random.seed(42)
pd.set_option("display.max_columns", None)



## 2. Criando uma base sintética de clientes

Vamos criar 20.000 clientes com características típicas de uma base comercial:

- idade
- tempo como cliente
- frequência de compras
- recência
- ticket médio
- número de categorias compradas
- desconto médio
- canal digital
- tratamento: recebeu ou não a campanha
- outcome: comprou ou não depois da campanha

Também vamos criar um **efeito causal verdadeiro oculto** nos dados. Isso permite comparar o uplift estimado pelo modelo com o efeito que realmente gerou os dados.


In [ ]:

n = 20000

df = pd.DataFrame({
    "idade": np.random.randint(18, 76, n),
    "tempo_cliente_meses": np.random.randint(1, 121, n),
    "frequencia_12m": np.random.poisson(8, n).clip(0, 40),
    "recencia_dias": np.random.gamma(2.5, 35, n).clip(1, 365),
    "ticket_medio": np.random.lognormal(mean=np.log(250), sigma=0.65, size=n).clip(30, 3000),
    "categorias_12m": np.random.poisson(3, n).clip(1, 12),
    "desconto_medio": np.random.beta(2, 8, n),
    "canal_digital": np.random.binomial(1, 0.65, n)
})

# Probabilidade base de compra sem campanha
logit_base = (
    -2.2
    + 0.10 * df["frequencia_12m"]
    - 0.008 * df["recencia_dias"]
    + 0.0009 * df["ticket_medio"]
    + 0.12 * df["categorias_12m"]
    + 0.35 * df["canal_digital"]
    + 0.15 * np.log1p(df["tempo_cliente_meses"])
)

p_base = 1 / (1 + np.exp(-logit_base))
p_base = np.clip(p_base, 0.01, 0.90)

# Efeito causal heterogêneo:
# clientes com maior frequência e recência menor respondem melhor à campanha
uplift_true = (
    0.02
    + 0.055 * (df["frequencia_12m"] / 10)
    - 0.00008 * df["recencia_dias"]
    + 0.04 * df["canal_digital"]
    + 0.03 * (1 - df["desconto_medio"])
)

uplift_true = np.clip(uplift_true, -0.05, 0.35)

# Randomização do tratamento: 50% recebem a campanha
df["treatment"] = np.random.binomial(1, 0.5, n)

# Probabilidade final depende do tratamento
p_treated = np.clip(p_base + uplift_true, 0.01, 0.97)
p_outcome = np.where(df["treatment"] == 1, p_treated, p_base)

df["comprou"] = np.random.binomial(1, p_outcome)

# Guardamos o efeito verdadeiro apenas para avaliação didática
df["uplift_true"] = uplift_true

df.head()


## 3. Conhecendo os dados

In [ ]:

print("Shape:", df.shape)
print("\nTratamento:")
print(df["treatment"].value_counts(normalize=True))

print("\nTaxa de compra por grupo:")
print(df.groupby("treatment")["comprou"].mean())

print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(
    data=df, x="recencia_dias", hue="treatment",
    kde=True, ax=axes[0], element="step"
)
axes[0].set_title("Distribuição da Recência")

sns.histplot(
    data=df, x="frequencia_12m", hue="treatment",
    kde=True, ax=axes[1], element="step"
)
axes[1].set_title("Distribuição da Frequência")

plt.tight_layout()
plt.show()



## 4. Efeito médio da campanha

Antes do ML, precisamos entender o efeito agregado.

O **ATE (Average Treatment Effect)**, neste experimento randomizado, pode ser estimado por:

`taxa de compra no tratamento - taxa de compra no controle`


In [ ]:

rate_treatment = df.loc[df["treatment"] == 1, "comprou"].mean()
rate_control = df.loc[df["treatment"] == 0, "comprou"].mean()

ate = rate_treatment - rate_control

print(f"Taxa de compra - Tratamento: {rate_treatment:.2%}")
print(f"Taxa de compra - Controle:   {rate_control:.2%}")
print(f"ATE / uplift médio:          {ate:.2%}")



## 5. Modelo tradicional de propensão

Primeiro vamos fazer algo que seria comum em um projeto de ML tradicional:

> prever a probabilidade de compra usando apenas características do cliente.

O tratamento **não entra como feature** aqui. O objetivo é mostrar que alta propensão não significa necessariamente alto impacto da campanha.


In [ ]:

features = [
    "idade", "tempo_cliente_meses", "frequencia_12m",
    "recencia_dias", "ticket_medio", "categorias_12m",
    "desconto_medio", "canal_digital"
]

X = df[features]
y = df["comprou"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

propensity_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=30,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

propensity_model.fit(X_train, y_train)

df.loc[X_test.index, "propensity_score"] = propensity_model.predict_proba(X_test)[:, 1]

auc_propensity = roc_auc_score(y_test, df.loc[X_test.index, "propensity_score"])
print(f"ROC-AUC do modelo de propensão: {auc_propensity:.3f}")



## 6. Por que propensão não é suficiente?

Vamos olhar os clientes com maior propensão.

Um cliente pode ter 90% de chance de comprar **mesmo sem receber a campanha**. Nesse caso, gastar dinheiro para convencê-lo tem pouco valor incremental.

O uplift tenta estimar:

`P(compra | tratamento, X) - P(compra | controle, X)`



## 7. Uplift Modeling com T-Learner

O T-Learner cria dois modelos:

**Modelo T (tratamento):**

`X → P(Y=1 | T=1, X)`

**Modelo C (controle):**

`X → P(Y=1 | T=0, X)`

Depois:

`Uplift = P(Y=1 | T=1, X) - P(Y=1 | T=0, X)`

O resultado é uma estimativa do **CATE (Conditional Average Treatment Effect)** para cada cliente.


In [ ]:

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.30,
    random_state=42,
    stratify=df["treatment"]
)

train = df.loc[train_idx].copy()
test = df.loc[test_idx].copy()

treated_train = train[train["treatment"] == 1]
control_train = train[train["treatment"] == 0]

model_t = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model_c = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model_t.fit(treated_train[features], treated_train["comprou"])
model_c.fit(control_train[features], control_train["comprou"])

p1 = model_t.predict_proba(test[features])[:, 1]
p0 = model_c.predict_proba(test[features])[:, 1]

test["p_treatment"] = p1
test["p_control"] = p0
test["uplift"] = test["p_treatment"] - test["p_control"]

test[[
    "p_treatment", "p_control", "uplift"
]].head(10)


## 8. Ranking dos clientes por uplift

In [ ]:

ranking = test.sort_values("uplift", ascending=False).copy()

ranking[[
    "idade", "frequencia_12m", "recencia_dias",
    "ticket_medio", "p_treatment", "p_control", "uplift"
]].head(20)



## 9. Segmentando os clientes

Uma interpretação prática do uplift é dividir os clientes em quatro grupos:

1. **Persuadíveis:** baixa/menor probabilidade sem campanha e alto ganho com tratamento.
2. **Certos:** comprariam de qualquer maneira.
3. **Perdidos:** baixa probabilidade nos dois cenários.
4. **Adverse:** a campanha pode reduzir a probabilidade de compra.

Na prática, os thresholds devem ser definidos de acordo com o negócio e os custos da campanha.


In [ ]:

def classify_uplift(row):
    if row["uplift"] >= 0.08 and row["p_control"] < 0.65:
        return "Persuadível"
    elif row["p_control"] >= 0.65 and row["p_treatment"] >= 0.65:
        return "Certo"
    elif row["uplift"] < 0:
        return "Adverse"
    else:
        return "Perdido / Baixo impacto"

test["segmento_uplift"] = test.apply(classify_uplift, axis=1)

print(test["segmento_uplift"].value_counts())


In [ ]:

segment_summary = (
    test.groupby("segmento_uplift")
    .agg(
        clientes=("uplift", "size"),
        uplift_medio=("uplift", "mean"),
        propensao_com_campanha=("p_treatment", "mean"),
        propensao_sem_campanha=("p_control", "mean")
    )
    .sort_values("uplift_medio", ascending=False)
)

segment_summary



## 10. Comparando propensão e uplift

Agora podemos enxergar a diferença conceitual:

- **Propensão:** prioriza quem provavelmente compra.
- **Uplift:** prioriza quem provavelmente muda seu comportamento por causa da campanha.

Em um projeto real, a segunda abordagem pode reduzir desperdício de campanhas em clientes que comprariam de qualquer maneira.


In [ ]:

comparison = test.copy()

comparison["quartil_propensao"] = pd.qcut(
    comparison["p_treatment"], 10, labels=False, duplicates="drop"
)

comparison["quartil_uplift"] = pd.qcut(
    comparison["uplift"], 10, labels=False, duplicates="drop"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=comparison,
    x="quartil_propensao",
    y="uplift",
    ax=axes[0]
)
axes[0].set_title("Uplift dentro dos decis de propensão")
axes[0].set_xlabel("Decil de propensão")

sns.histplot(
    comparison["uplift"],
    bins=40,
    kde=True,
    ax=axes[1]
)
axes[1].axvline(0, linestyle="--")
axes[1].set_title("Distribuição do Uplift estimado")
axes[1].set_xlabel("Uplift")

plt.tight_layout()
plt.show()



## 11. Avaliação simples do ranking de uplift

Como não observamos simultaneamente o resultado tratado e não tratado para o mesmo cliente, não podemos calcular o uplift individual diretamente.

Mas, como estamos trabalhando com um experimento sintético/randomizado, podemos avaliar o ranking por grupos.

Vamos ordenar os clientes pelo uplift previsto e observar o efeito incremental acumulado quando direcionamos a campanha para os maiores scores.


In [ ]:

eval_df = test.copy().sort_values("uplift", ascending=False).reset_index(drop=True)

eval_df["rank"] = np.arange(1, len(eval_df) + 1)
eval_df["decile"] = pd.qcut(
    eval_df["rank"],
    10,
    labels=[f"D{i}" for i in range(1, 11)]
)

decile_results = []

for decile, group in eval_df.groupby("decile", observed=False):
    treated = group[group["treatment"] == 1]
    control = group[group["treatment"] == 0]

    if len(treated) > 0 and len(control) > 0:
        observed_uplift = (
            treated["comprou"].mean() -
            control["comprou"].mean()
        )
    else:
        observed_uplift = np.nan

    decile_results.append({
        "decile": decile,
        "clientes": len(group),
        "uplift_previsto": group["uplift"].mean(),
        "uplift_observado": observed_uplift
    })

decile_results = pd.DataFrame(decile_results)
decile_results


In [ ]:

plt.figure(figsize=(10, 5))

plt.plot(
    decile_results["decile"].astype(str),
    decile_results["uplift_previsto"],
    marker="o",
    label="Uplift previsto"
)

plt.plot(
    decile_results["decile"].astype(str),
    decile_results["uplift_observado"],
    marker="o",
    label="Uplift observado"
)

plt.axhline(0, linestyle="--")
plt.title("Uplift previsto vs. observado por decil")
plt.xlabel("Decil do ranking")
plt.ylabel("Uplift")
plt.legend()
plt.tight_layout()
plt.show()



## 12. Simulação de targeting

Vamos comparar uma estratégia baseada em propensão com uma estratégia baseada em uplift.

A pergunta é:

> Se eu só puder abordar 20% da minha base, qual estratégia encontra mais efeito incremental?

A estratégia de propensão seleciona os clientes com maior probabilidade de compra.

A estratégia de uplift seleciona os clientes com maior efeito incremental estimado.


In [ ]:

target_fraction = 0.20
k = int(len(test) * target_fraction)

prop_target = test.nlargest(k, "p_treatment").copy()
uplift_target = test.nlargest(k, "uplift").copy()

def observed_incremental_effect(group):
    treated = group[group["treatment"] == 1]
    control = group[group["treatment"] == 0]

    if len(treated) == 0 or len(control) == 0:
        return np.nan

    return treated["comprou"].mean() - control["comprou"].mean()

prop_effect = observed_incremental_effect(prop_target)
uplift_effect = observed_incremental_effect(uplift_target)

print(f"Clientes selecionados: {k:,} ({target_fraction:.0%} da base de teste)")
print(f"Uplift observado - targeting por propensão: {prop_effect:.2%}")
print(f"Uplift observado - targeting por uplift:    {uplift_effect:.2%}")



## 13. Valor financeiro da campanha

Uma aplicação de negócio precisa considerar custo e receita.

Suponha:

- margem média incremental por compra: R$ 250
- custo da campanha por cliente: R$ 10

Podemos estimar um valor incremental aproximado:

`clientes alvo × uplift × margem - custo da campanha`

Esta é uma simplificação didática. Em uma aplicação real, seria necessário considerar margem real, custos variáveis, descontos, capacidade comercial e outros fatores.


In [ ]:

margem_incremental = 250
custo_campanha = 10

def estimated_campaign_value(target):
    incremental_sales = target["uplift"].clip(lower=0).sum()
    revenue_value = incremental_sales * margem_incremental
    campaign_cost = len(target) * custo_campanha
    return revenue_value - campaign_cost

value_propensity = estimated_campaign_value(prop_target)
value_uplift = estimated_campaign_value(uplift_target)

print(f"Valor estimado - targeting por propensão: R$ {value_propensity:,.2f}")
print(f"Valor estimado - targeting por uplift:    R$ {value_uplift:,.2f}")



## 14. O que aprendemos?

### Propensity Model

Responde:

> **Quem provavelmente compra?**

É útil para:

- priorização comercial
- recomendação
- previsão de demanda individual
- campanhas de aquisição

### Uplift Model

Responde:

> **Quem provavelmente compra por causa da intervenção?**

É útil para:

- campanhas de marketing
- descontos
- retenção
- cross-sell
- upsell
- priorização de vendedores
- alocação de orçamento

### Causal ML

Vai além da correlação e tenta estimar o efeito de uma intervenção.

A ideia fundamental é o **counterfactual**:

> O que teria acontecido com este mesmo cliente se ele não tivesse recebido a campanha?

Não observamos os dois mundos simultaneamente. Por isso desenho experimental, randomização e hipóteses causais são fundamentais.



## 15. Próximos passos para um projeto real

Este notebook é propositalmente simples. Em um projeto profissional eu acrescentaria:

1. **A/B test bem desenhado**
2. Definição correta da janela de tratamento
3. Controle de leakage
4. Feature engineering temporal
5. Cross-validation apropriada
6. Qini Curve
7. Qini Coefficient
8. AUUC — Area Under the Uplift Curve
9. CATE
10. Doubly Robust Learner
11. Causal Forest
12. análise de custos e ROI
13. monitoramento do modelo
14. re-treinamento
15. integração com CRM

> **Regra prática:** não use uplift para simplesmente "achar segmentos". O objetivo é estimar **efeito incremental de uma ação** e transformar essa estimativa em uma decisão de negócio.
